In [2]:
# create_ecommerce_db.py

from sqlalchemy import create_engine, text


db_path = "sqlite:////Users/linqibin/Documents/code/agents/my-agents/apps/backend/db/ecommerce.sqlite"

engine = create_engine(
    db_path
)


with engine.connect() as conn:

    # 用户表
    conn.execute(text("""
    CREATE TABLE users (
        id INTEGER PRIMARY KEY,
        name TEXT,
        age INTEGER,
        city TEXT
    )
    """))


    # 商品表
    conn.execute(text("""
    CREATE TABLE products (
        id INTEGER PRIMARY KEY,
        name TEXT,
        category TEXT,
        price FLOAT
    )
    """))


    # 订单表
    conn.execute(text("""
    CREATE TABLE orders (
        id INTEGER PRIMARY KEY,
        user_id INTEGER,
        product_id INTEGER,
        quantity INTEGER,
        total_amount FLOAT,
        order_date TEXT
    )
    """))


    # 插入数据
    conn.execute(text("""
    INSERT INTO users VALUES
    (1,'张三',28,'北京'),
    (2,'李四',32,'上海'),
    (3,'王五',25,'深圳')
    """))


    conn.execute(text("""
    INSERT INTO products VALUES
    (1,'MacBook Pro','电脑',14999),
    (2,'iPhone 17','手机',8999),
    (3,'AirPods','耳机',1999)
    """))


    conn.execute(text("""
    INSERT INTO orders VALUES
    (1,1,1,1,14999,'2026-01-01'),
    (2,2,2,2,17998,'2026-01-02'),
    (3,3,3,3,5997,'2026-01-03')
    """))


    conn.commit()


print("数据库创建完成")

数据库创建完成


In [3]:
from langchain_community.utilities import SQLDatabase


db = SQLDatabase.from_uri(
    db_path
)


print(
    db.get_usable_table_names()
)


print(
    db.get_table_info()
)

['orders', 'products', 'users']

CREATE TABLE orders (
	id INTEGER, 
	user_id INTEGER, 
	product_id INTEGER, 
	quantity INTEGER, 
	total_amount FLOAT, 
	order_date TEXT, 
	PRIMARY KEY (id)
)

/*
3 rows from orders table:
id	user_id	product_id	quantity	total_amount	order_date
1	1	1	1	14999.0	2026-01-01
2	2	2	2	17998.0	2026-01-02
3	3	3	3	5997.0	2026-01-03
*/


CREATE TABLE products (
	id INTEGER, 
	name TEXT, 
	category TEXT, 
	price FLOAT, 
	PRIMARY KEY (id)
)

/*
3 rows from products table:
id	name	category	price
1	MacBook Pro	电脑	14999.0
2	iPhone 17	手机	8999.0
3	AirPods	耳机	1999.0
*/


CREATE TABLE users (
	id INTEGER, 
	name TEXT, 
	age INTEGER, 
	city TEXT, 
	PRIMARY KEY (id)
)

/*
3 rows from users table:
id	name	age	city
1	张三	28	北京
2	李四	32	上海
3	王五	25	深圳
*/


/var/folders/r2/ww1nfn0x20j21k3nmn17w4900000gn/T/ipykernel_8512/3680429403.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.sql_database.query import create_sql_query_chain

from dotenv import load_dotenv

load_dotenv()

import os

model = os.getenv('MODEL_NAME', '')

llm = ChatOpenAI(
    model=model,
    temperature=0,
    extra_body={'enbale_thinking': False}
)

prompt = PromptTemplate.from_template(
"""
你是一个专业SQL生成助手。

你的任务：

根据用户问题生成SQLite SQL。


规则：

1. 只输出SQL
2. 不要解释
3. 不要Markdown代码块
4. 只能生成SELECT语句
5. 使用数据库已有字段
6. 限制返回数量为 {top_k}


数据库结构:

{table_info}


用户问题:

{input}


SQL:

"""
)


In [16]:
chain = create_sql_query_chain(
    llm,
    db,
    prompt=prompt,
    k=1
)

In [17]:
question = """
查询销售额最高的商品
"""


sql = chain.invoke(
    {
        "question": question
    }
)


print("===================")
print(sql)
print("===================")

SELECT p.name, SUM(o.total_amount) FROM orders o JOIN products p ON o.product_id = p.id GROUP BY p.id, p.name ORDER BY SUM(o.total_amount) DESC LIMIT 1


In [18]:
engine = create_engine(
    db_path
)


# 测试SQL
# sql = """
# SELECT *
# FROM users
# LIMIT 10;
# """


def execute_sql(sql: str):

    with engine.connect() as conn:

        result = conn.execute(
            text(sql)
        )

        rows = result.fetchall()

        # 获取字段名
        columns = result.keys()

        print("字段:")
        print(list(columns))

        print("\n数据:")

        for row in rows:
            print(dict(row._mapping))


if __name__ == "__main__":

    execute_sql(sql)

字段:
['name', 'SUM(o.total_amount)']

数据:
{'name': 'iPhone 17', 'SUM(o.total_amount)': 17998.0}
